In [1]:
import networkx as nx
import geopandas as gpd
import numpy as np
from libpysal import graph
import warnings
warnings.filterwarnings('ignore')
from joblib import Parallel, delayed
import os

In [2]:
gdfs_sac = ["elections","guerry","sacramento"]

In [3]:
gdfs_reg = ["south","chicago", "ceara_zika"]

In [4]:
def graph_to_edge_set(g, unique_ids):
    """
    Convert a libpysal Graph to upper-triangle (i < j) integer index pairs
    matching the sparse matrix row/column indices.
    """
    id_to_int = {uid: idx for idx, uid in enumerate(unique_ids)}
    adj = g.adjacency.reset_index()

    i_series = adj['focal'].map(id_to_int)
    j_series = adj['neighbor'].map(id_to_int)

    valid = i_series.notna() & j_series.notna()
    i_arr = i_series[valid].astype(int).values
    j_arr = j_series[valid].astype(int).values

    mask = i_arr != j_arr
    i_arr, j_arr = i_arr[mask], j_arr[mask]

    upper_i = np.minimum(i_arr, j_arr)
    upper_j = np.maximum(i_arr, j_arr)

    return list(set(zip(upper_i.tolist(), upper_j.tolist())))
    
def get_deletable_edges(W_sparse):
    """Edges where both endpoints have degree > 1 (no isolates created)."""
    W_sparse = W_sparse.tocsr()
    deletable = []
    for i in range(W_sparse.shape[0]):
        degree_i = W_sparse.getrow(i).nnz - (1 if W_sparse[i, i] != 0 else 0)
        if degree_i > 1:
            row = W_sparse.getrow(i).tocoo()
            for j in row.col:
                if i < j:
                    degree_j = W_sparse.getcol(j).nnz - (1 if W_sparse[j, j] != 0 else 0)
                    if degree_j > 1:
                        deletable.append((i, j))
    return deletable


def get_deletable_edges_mst(W_sparse, max_components):
    """Edges safe to delete without exceeding max_components."""
    G = nx.from_scipy_sparse_array(W_sparse)
    current_components = nx.number_connected_components(G)
    all_edges = set(G.edges())
    non_bridges = all_edges - set(nx.bridges(G))

    if current_components >= max_components:
        return list(non_bridges)
    return list(all_edges)

In [5]:
def build_g_borders(gdf, g_rook, region_col):
    g_block = graph.Graph.build_block_contiguity(gdf[region_col])
    g_borders = g_rook.difference(g_block)
    g_remaining = g_rook.difference(g_borders)
    return g_borders, g_remaining


def build_g_region(gdf, g_rook, region_col, region_id=None):
    if region_id is None:
        region_id = np.random.choice(gdf[region_col].unique())
    region_ids = gdf[gdf[region_col] == region_id].index
    edges_df = g_rook.adjacency.reset_index()
    mask = edges_df['focal'].isin(region_ids) | edges_df['neighbor'].isin(region_ids)
    edges_region = edges_df[mask]
    g_region = graph.Graph.from_adjacency(edges_region, 'focal', 'neighbor', 'weight')
    g_remaining = g_rook.difference(g_region)
    return g_region, g_remaining

In [6]:
def remove_one_edge(W_sparse, method, max_components=None, candidate_edges=None):
    """
    Remove a single random deletable edge.

    Parameters
    ----------
    method          : 'no_isolates' | 'mst'
    max_components  : int, required when method='mst'
    candidate_edges : list of (i, j) upper-triangle pairs to restrict deletion to
    """
    if method == "no_isolates":
        deletable = get_deletable_edges(W_sparse)
    else:
        deletable = get_deletable_edges_mst(W_sparse, max_components=max_components)

    if candidate_edges is not None:
        candidate_set = set(candidate_edges)
        deletable = [(i, j) for i, j in deletable if (i, j) in candidate_set]

    if len(deletable) == 0:
        return W_sparse.copy(), False

    i, j = deletable[np.random.randint(len(deletable))]
    W_new = W_sparse.tolil()
    W_new[i, j] = 0
    W_new[j, i] = 0
    return W_new.tocsr(), True

def run_corruption_loop(W_sparse, method, max_components=None, candidate_edges=None):
    """
    Remove edges one at a time until no more safe deletable edges remain.
    Returns ordered list of Graphs — index i has i+1 edges removed.
    """
    W = W_sparse.copy()
    corrupted_graphs = []
    while True:
        W, success = remove_one_edge(
            W, method=method,
            max_components=max_components,
            candidate_edges=candidate_edges,
        )
        if not success:
            break
        corrupted_graphs.append(graph.Graph.from_sparse(W))
    return corrupted_graphs

In [7]:
def corrupt_run_random(name, n, method):
    
    import os
    import tools
    from libpysal import graph

    os.makedirs(f"graphs/real/{name}/{method}/random", exist_ok=True)
    g_true = graph.read_parquet(f"graphs/real/{name}/g_true.parquet")
    W_sparse = g_true.to_W().sparse

    if method == "one_component":
        kwargs = dict(method='mst', max_components=1)
    else:  # no_isolates
        kwargs = dict(method='no_isolates')

    corrupted_graphs = run_corruption_loop(W_sparse, **kwargs)

    for i, corrupted_g in enumerate(corrupted_graphs, start=1):
        corrupted_g.to_parquet(
            f"graphs/real/{name}/{method}/random/g_missing{i}_run{n}.parquet"
        )

In [7]:
n_corrupt = 10

In [8]:
tasks = [(name, n) for name in gdfs_sac for n in range(n_corrupt)]
Parallel(n_jobs=-1)(delayed(corrupt_run_random)(name, n, "no_isolates") for name, n in tasks)

[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None]

In [9]:
tasks = [(name, n) for name in gdfs_reg for n in range(n_corrupt)]
Parallel(n_jobs=-1)(delayed(corrupt_run_random)(name, n, "one_component") for name, n in tasks)

[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None]

In [8]:
def corrupt_borders_local(name, gdf, region_col, condition, n_border_runs=10, n_local_runs=2):
    # borders and local corruption
    kwargs = dict(method='mst', max_components=1) if condition == "one_component" else dict(method='no_isolates')
    g_true = graph.read_parquet(f"graphs/real/{name}/g_true.parquet")
    W_sparse = g_true.to_W().sparse
    
    # borders: 10 runs, same fixed region column each time
    g_borders, _ = build_g_borders(gdf, g_true, region_col=region_col)
    border_edges = graph_to_edge_set(g_borders, g_true.unique_ids)
    borders_dir = f"graphs/real/{name}/{condition}/borders"
    os.makedirs(borders_dir, exist_ok=True)
    for n in range(n_border_runs):
        corrupted_graphs = run_corruption_loop(W_sparse, candidate_edges=border_edges, **kwargs)
        for i, corrupted_g in enumerate(corrupted_graphs, start=1):
            corrupted_g.to_parquet(f"{borders_dir}/g_missing{i}_run{n}.parquet")
            
    # local: n_local_runs per unique region value
    local_dir = f"graphs/real/{name}/{condition}/local"
    os.makedirs(local_dir, exist_ok=True)
    for region_id in gdf[region_col].unique():
        region_str = str(int(region_id)) if isinstance(region_id, float) else str(region_id)
        g_region, _ = build_g_region(gdf, g_true, region_col=region_col, region_id=region_id)
        local_edges = graph_to_edge_set(g_region, g_true.unique_ids)
        for n in range(n_local_runs):
            corrupted_graphs = run_corruption_loop(W_sparse, candidate_edges=local_edges, **kwargs)
            for i, corrupted_g in enumerate(corrupted_graphs, start=1):
                corrupted_g.to_parquet(f"{local_dir}/g_missing{i}_region_{region_str}_run{n}.parquet")

In [4]:
guerry_gdf = gpd.read_parquet("data/real/guerry.parquet")
south_gdf = gpd.read_parquet("data/real/south.parquet")

corrupt_borders_local("guerry", guerry_gdf, region_col="Region", condition="no_isolates",n_local_runs=2)
corrupt_borders_local("south", south_gdf, region_col="STATE_FIPS", condition="one_component",n_local_runs=1)